# RAG (Retrieval-Augmented Generation) Using LangChain v1

This notebook builds a RAG pipeline using LangChain v1 (the current version).

**Why was the old notebook broken?**

LangChain v1 removed `langchain.chains` (including `RetrievalQA`) from the core package.
These legacy chains were moved to a separate package called `langchain-classic`.
This notebook uses the modern LCEL approach instead, which is the recommended way in v1.

**RAG Flow:**
```
Documents --> Split into Chunks --> Embed Chunks --> Store in VectorDB
Question  --> Embed Question   --> Retrieve Relevant Chunks --> GPT-4o Mini answers
```

---

## Step 1 - Install Required Libraries

Note: We no longer need `langchain-community` for FAISS in v1.
We install `langchain-community` only for the FAISS vector store wrapper.

In [ ]:
# Install all required packages
# langchain         - core package (v1)
# langchain-openai  - OpenAI integrations
# langchain-community - community integrations (FAISS wrapper)
# faiss-cpu         - the FAISS vector store
# tiktoken          - tokenizer used by OpenAI models

!pip install langchain langchain-openai langchain-community faiss-cpu tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## Step 2 - Set Your OpenAI API Key

In [ ]:
import os

# Set your OpenAI API key here
os.environ["OPENAI_API_KEY"] = ""

print("API key set.")

API key set.


## Step 3 - Import All Required Modules

**What changed from v0 to v1:**

| Old (broken in v1) | New (correct in v1) |
|---|---|
| `from langchain.chains import RetrievalQA` | Not needed - use LCEL pipeline instead |
| `from langchain.schema import Document` | `from langchain_core.documents import Document` |
| `from langchain.text_splitter import ...` | `from langchain_text_splitters import ...` |

In [ ]:
# Correct imports for LangChain v1
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("All modules imported successfully.")

All modules imported successfully.


/tmp/ipykernel_3405/2041565274.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## Step 4 - Prepare Your Documents

In a real project, you would load PDFs or text files.
Here we use simple strings to keep things clear and focused.

In [ ]:
# These are the documents our RAG system will use to answer questions
raw_documents = [
    """LangChain is a framework for developing applications powered by language models.
    It provides tools to connect LLMs with external data sources and APIs.
    In LangChain v1, legacy chains were removed from the core package.
    The modern approach uses LCEL (LangChain Expression Language) to build pipelines.""",

    """RAG stands for Retrieval-Augmented Generation.
    It is a technique where a model retrieves relevant documents before generating an answer.
    RAG helps the model answer questions about data it was not trained on.
    This reduces hallucinations and keeps answers grounded in real content.""",

    """OpenAI provides powerful language models like GPT-4o and GPT-4o Mini.
    GPT-4o Mini is a smaller, faster, and more cost-efficient version of GPT-4o.
    It is well suited for tasks that do not require the full power of GPT-4o.
    OpenAI also provides the text-embedding-ada-002 model for creating embeddings.""",

    """FAISS is a library developed by Facebook AI for efficient similarity search.
    It is commonly used as a vector store in RAG pipelines.
    FAISS stores embeddings and retrieves the most similar ones for a given query.
    In LangChain, FAISS is available through the langchain-community package."""
]

# Wrap each string into a LangChain Document object
documents = [Document(page_content=text) for text in raw_documents]

print(f"Loaded {len(documents)} documents.")

Loaded 4 documents.


## Step 5 - Split Documents into Smaller Chunks

Large documents are split into smaller chunks so the retriever can find
only the most relevant pieces, rather than loading everything at once.

In [ ]:
# chunk_size   : max characters per chunk
# chunk_overlap: characters that overlap between adjacent chunks
#                (preserves context at chunk boundaries)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=30
)

chunks = text_splitter.split_documents(documents)

print(f"Split into {len(chunks)} chunks.")
print("\nExample chunk:")
print(chunks[0].page_content)

Split into 16 chunks.

Example chunk:
LangChain is a framework for developing applications powered by language models.


## Step 6 - Create Embeddings and Build a Vector Store

Embeddings convert text into numerical vectors.
FAISS stores these vectors and enables fast similarity search.

In [ ]:
# Use OpenAI text-embedding-ada-002 to create embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Build the FAISS vector store from our document chunks
# This embeds each chunk and stores the vectors in memory
vector_store = FAISS.from_documents(chunks, embeddings)

print("Vector store created successfully.")
print(f"Total vectors stored: {vector_store.index.ntotal}")

Vector store created successfully.
Total vectors stored: 16


## Step 7 - Set Up the Retriever

The retriever takes a question, embeds it, and fetches the top-k
most similar chunks from the vector store.

In [ ]:
# k=2 means retrieve the 2 most relevant chunks per question
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# Quick test
test_chunks = retriever.invoke("What is RAG?")
print(f"Retrieved {len(test_chunks)} chunks for: 'What is RAG?'")
print("\nChunk 1:")
print(test_chunks[0].page_content)

Retrieved 2 chunks for: 'What is RAG?'

Chunk 1:
RAG stands for Retrieval-Augmented Generation.
    It is a technique where a model retrieves relevant documents before generating an answer.


## Step 8 - Initialize GPT-4o Mini

In [ ]:
# temperature=0 makes the model deterministic
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("GPT-4o Mini initialized.")

GPT-4o Mini initialized.


## Step 9 - Define the Prompt Template

The prompt instructs the model to answer only based on the provided context.
This keeps answers grounded in our documents.

In [ ]:
# This is the prompt that gets sent to GPT-4o Mini
# {context} will be filled with the retrieved chunks
# {question} will be filled with the user's question

prompt_template = ChatPromptTemplate.from_template("""
You are a helpful assistant. Answer the question using only the context below.
If the answer is not in the context, say "I do not have enough information to answer that."

Context:
{context}

Question: {question}

Answer:
""")

print("Prompt template defined.")

Prompt template defined.


## Step 10 - Build the RAG Chain Using LCEL

LCEL (LangChain Expression Language) uses the `|` pipe operator to chain steps together.
This replaces the old `RetrievalQA` chain that no longer exists in LangChain v1.

**Pipeline:**
```
question --> retriever --> format context --> prompt --> GPT-4o Mini --> answer
```

In [ ]:
def format_docs(docs):
    """Join all retrieved chunks into a single context string."""
    return "\n\n".join(doc.page_content for doc in docs)


# Build the RAG chain using the LCEL pipe operator
# Each step feeds into the next:
#
#  Step A: Retrieve relevant chunks (retriever) + pass question through (RunnablePassthrough)
#  Step B: Fill the prompt template with context and question
#  Step C: Send the prompt to GPT-4o Mini
#  Step D: Parse the output as a plain string

rag_chain = (
    {
        "context": retriever | format_docs,   # Retrieve chunks, then join them
        "question": RunnablePassthrough()      # Pass the question through unchanged
    }
    | prompt_template   # Fill the prompt template
    | llm               # Call GPT-4o Mini
    | StrOutputParser() # Extract the string answer
)

print("RAG chain built successfully using LCEL.")

RAG chain built successfully using LCEL.


## Step 11 - Ask Questions

The chain retrieves relevant chunks and uses GPT-4o Mini to generate a grounded answer.

In [ ]:
def ask(question):
    """Ask a question and display the answer along with the source chunks used."""
    print(f"Question: {question}")
    print("-" * 60)

    # Get the answer from the RAG chain
    answer = rag_chain.invoke(question)
    print(f"Answer: {answer}")

    # Also show which chunks were retrieved (for transparency)
    retrieved = retriever.invoke(question)
    print("\nSource chunks used:")
    for i, doc in enumerate(retrieved, 1):
        print(f"  [{i}] {doc.page_content[:100]}...")
    print("=" * 60)

In [ ]:
# Question 1
ask("What is RAG and why is it useful?")

Question: What is RAG and why is it useful?
------------------------------------------------------------
Answer: RAG stands for Retrieval-Augmented Generation. It is a technique where a model retrieves relevant documents before generating an answer. RAG is useful because it helps the model answer questions about data it was not trained on, reducing hallucinations and keeping answers grounded in real content.

Source chunks used:
  [1] RAG stands for Retrieval-Augmented Generation.
    It is a technique where a model retrieves relevan...
  [2] RAG helps the model answer questions about data it was not trained on.
    This reduces hallucinatio...


In [ ]:
# Question 2
ask("What changed in LangChain v1 regarding chains?")

Question: What changed in LangChain v1 regarding chains?
------------------------------------------------------------
Answer: In LangChain v1, legacy chains were removed from the core package, and the modern approach now uses LCEL (LangChain Expression Language) to build pipelines.

Source chunks used:
  [1] In LangChain v1, legacy chains were removed from the core package.
    The modern approach uses LCEL...
  [2] LangChain is a framework for developing applications powered by language models.
    It provides too...


In [ ]:
# Question 3
ask("What is FAISS and where does it come from?")

Question: What is FAISS and where does it come from?
------------------------------------------------------------
Answer: FAISS is a library developed by Facebook AI for efficient similarity search.

Source chunks used:
  [1] FAISS is a library developed by Facebook AI for efficient similarity search.
    It is commonly used...
  [2] FAISS stores embeddings and retrieves the most similar ones for a given query.
    In LangChain, FAI...


In [ ]:
# Question 4 - outside our documents, should say it does not know
ask("Who is the CEO of OpenAI?")

Question: Who is the CEO of OpenAI?
------------------------------------------------------------
Answer: I do not have enough information to answer that.

Source chunks used:
  [1] OpenAI provides powerful language models like GPT-4o and GPT-4o Mini.
    GPT-4o Mini is a smaller, ...
  [2] It is well suited for tasks that do not require the full power of GPT-4o.
    OpenAI also provides t...
